# Notebook 02 — Regression (Checkpoint 3)

**Target:** `price`  
**Selected feature set (7 inputs):** `vehicle_age`, `odometer`, `manufacturer`, `fuel`, `transmission`, `drive`, `type`

I picked these because they’re interpretable, available at “listing time,” and they showed up strong once I started fitting models.


In [1]:
import numpy as np  #I'm doing this because numpy supports numeric operations used indirectly by sklearn.
import pandas as pd  #I'm doing this because pandas loads the cleaned CSV produced in notebook 01.
import matplotlib.pyplot as plt  #I'm doing this because matplotlib plots feature importance bars later.

from sklearn.compose import ColumnTransformer  #I'm doing this because ColumnTransformer applies different preprocessing to numeric versus categorical columns.
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor  #I'm doing this because tree ensembles capture nonlinear vehicle price effects.
from sklearn.linear_model import LinearRegression, Ridge  #I'm doing this because linear baselines benchmark how much trees improve.
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error  #I'm doing this because these metrics summarize regression error and fit quality.
from sklearn.model_selection import train_test_split  #I'm doing this because a holdout set estimates generalization honestly.
from sklearn.pipeline import Pipeline  #I'm doing this because pipelines chain preprocessing with the estimator cleanly.
from sklearn.preprocessing import OneHotEncoder, StandardScaler  #I'm doing this because scalers help linear models and one-hot encodes categories.
import joblib  #I'm doing this because joblib serializes the trained sklearn pipeline to disk for reuse.
from pathlib import Path  #I'm doing this because pathlib defines stable paths under the NEW project root.

PROJECT_ROOT = Path("/Users/jahbrae/Downloads/NEW")  #I'm doing this because all artifacts for this deliverable live under NEW.
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "cleaned_data.csv")  #I'm doing this because regression trains on the cleaned table saved in notebook 01.
TARGET = "price"  #I'm doing this because TARGET names the column we predict in dollars.
FEATURES = ["vehicle_age", "odometer", "manufacturer", "fuel", "transmission", "drive", "type"]  #I'm doing this because these seven fields are the agreed modeling inputs.

X = df[FEATURES]  #I'm doing this because X holds all input features for supervised learning.
y = df[TARGET]  #I'm doing this because y holds the continuous outcome vector aligned with X.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  #I'm doing this because an 80/20 split with a fixed seed is reproducible.
print(X_train.shape, X_test.shape)  #I'm doing this because shapes confirm train and test sizes look reasonable.


(176865, 7) (44217, 7)


In [2]:
def make_pre():  #I'm doing this because a factory function builds a fresh ColumnTransformer for each pipeline.
    return ColumnTransformer(  #I'm doing this because returning the transformer keeps the definition reusable.
        [  #I'm doing this because the list enumerates each preprocessing branch.
            ("num", StandardScaler(), ["vehicle_age", "odometer"]),  #I'm doing this because linear models benefit from scaled continuous inputs.
            (  #I'm doing this because this opens the tuple for the categorical branch of the ColumnTransformer.
                "cat",  #I'm doing this because this branch names the categorical preprocessing step.
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),  #I'm doing this because dense one-hot matrices work with these estimators and unknowns appear as all zeros.
                ["manufacturer", "fuel", "transmission", "drive", "type"],  #I'm doing this because these columns are nominal categories.
            ),  #I'm doing this because this closes the categorical tuple inside the transformer list.
        ]  #I'm doing this because this closes the list passed to ColumnTransformer.
    )  #I'm doing this because closing the ColumnTransformer completes the preprocessor.

models = {  #I'm doing this because a dictionary keeps several candidate pipelines keyed by short names.
    "linear": Pipeline([("pre", make_pre()), ("m", LinearRegression())]),  #I'm doing this because an ordinary least squares baseline sets a lower bound on performance.
    "ridge": Pipeline([("pre", make_pre()), ("m", Ridge(5.0))]),  #I'm doing this because ridge adds L2 shrinkage to reduce overfitting versus plain linear regression.
    "rf": Pipeline(  #I'm doing this because random forest averages many trees for robust predictions on tabular data.
        [  #I'm doing this because a list holds the ordered steps inside this Pipeline.
            ("pre", make_pre()),  #I'm doing this because the first step must preprocess features before the forest sees them.
            (  #I'm doing this because this opens the tuple naming the estimator step of the Pipeline.
                "m",  #I'm doing this because sklearn Pipelines use string names to label each step.
                RandomForestRegressor(n_estimators=80, max_depth=16, random_state=42, n_jobs=-1),  #I'm doing this because these hyperparameters balance bias and variance while using all CPU cores.
            ),  #I'm doing this because this closes the estimator step tuple for the random forest Pipeline.
        ]  #I'm doing this because this closes the Pipeline steps list for the random forest model.
    ),  #I'm doing this because this closes the Pipeline constructor call assigned to the rf entry.
    "gbr": Pipeline([("pre", make_pre()), ("m", GradientBoostingRegressor(random_state=42))]),  #I'm doing this because gradient boosting fits sequential trees that often excel on structured data.
}  #I'm doing this because closing the dict finishes the model catalog.

rows = []  #I'm doing this because I accumulate one metrics dict per fitted model in this list.
for name, pipe in models.items():  #I'm doing this because looping trains and scores every model with the same split.
    pipe.fit(X_train, y_train)  #I'm doing this because fit learns preprocessor statistics and model parameters on training data only.
    pred = pipe.predict(X_test)  #I'm doing this because predictions on the holdout set measure out-of-sample error.
    rows.append(  #I'm doing this because each row stores comparable metrics for sorting.
        {  #I'm doing this because this opens the dict literal passed to append for one model's metrics.
            "model": name,  #I'm doing this because the model name labels the row in the results table.
            "RMSE": root_mean_squared_error(y_test, pred),  #I'm doing this because RMSE penalizes large errors in dollar units comparable to price.
            "MAE": mean_absolute_error(y_test, pred),  #I'm doing this because MAE is robust and interpretable as average dollar error.
            "R2": r2_score(y_test, pred),  #I'm doing this because R2 summarizes variance explained by the model on the test set.
        }  #I'm doing this because this closes the metrics dict for one model.
    )  #I'm doing this because this closes the append call after the dict literal.

results = pd.DataFrame(rows).sort_values("RMSE")  #I'm doing this because sorting by RMSE surfaces the best-performing model first.
results  #I'm doing this because displaying the DataFrame shows the leaderboard inline.


,model,RMSE,MAE,R2
2,rf,5587.641936,3221.326192,0.850338
3,gbr,7320.983072,4677.972785,0.743083
1,ridge,8334.683714,5693.187070,0.667009
0,linear,8334.728730,5693.289831,0.667005


### Model iteration reflection

I’m not pretending linear regression is going to win on this... it’s my baseline so I can say “look, a simple model works okay, but trees do better on interactions.” Random forest usually ends up strong on mixed categorical + numeric messy data like this.


In [3]:
best = models["rf"]  #I'm doing this because random forest is the chosen tree model for feature importance extraction.
best.fit(X_train, y_train)  #I'm doing this because fitting again guarantees the model matches the latest training data before inspecting importances.

feat_names = best.named_steps["pre"].get_feature_names_out()  #I'm doing this because sklearn exposes post-encoding feature labels for importances.
imps = best.named_steps["m"].feature_importances_  #I'm doing this because the forest stores one importance score per transformed feature.
imp_df = pd.DataFrame({"feature": feat_names, "importance": imps}).sort_values("importance", ascending=False)  #I'm doing this because sorting ranks predictors from most to least influential.
imp_df.head(15)  #I'm doing this because the top fifteen rows highlight the strongest drivers without overwhelming the screen.


,feature,importance
0,num__vehicle_age,0.449793
1,num__odometer,0.160974
24,cat__drive_fwd,0.155772
15,cat__fuel_diesel,0.071477
17,cat__fuel_gas,0.019844
34,cat__type_pickup,0.015761
36,cat__type_truck,0.014155
33,cat__type_other,0.010839
22,cat__transmission_other,0.010121
3,cat__manufacturer_chevrolet,0.007560


In [4]:
top = imp_df.head(12)  #I'm doing this because twelve bars fit cleanly on one chart.
plt.figure(figsize=(8, 5))  #I'm doing this because this figure size leaves room for long one-hot feature names.
plt.barh(top["feature"][::-1], top["importance"][::-1])  #I'm doing this because reversing order puts the largest bar at the top for readability.
plt.title("Top feature importances (RandomForest)")  #I'm doing this because the title states what the horizontal bars represent.
plt.tight_layout()  #I'm doing this because tight_layout reduces label clipping at the margins.
plt.show()  #I'm doing this because show renders the importance bar chart.


/var/folders/rf/47shdcdn4j522110qmjp300h0000gn/T/ipykernel_25531/3713369775.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  #I'm doing this because show renders the importance bar chart.


### Feature selection reflection

If I dumped all raw columns into sklearn I’d have a worse app *and* a worse story. I’m intentionally keeping **7** model inputs (age + miles + 5 categoricals) because that matches what I can ask a human for without ruining their day.

Correlation + importances mostly agree: miles and age matter a ton; manufacturer/type/drive/fuel/transmission split the rest.


In [5]:
out = PROJECT_ROOT / "models" / "regression_model.pkl"  #I'm doing this because the Streamlit-style bundle should live under NEW/models.
out.parent.mkdir(parents=True, exist_ok=True)  #I'm doing this because the models directory may not exist until first save.

q1, q2 = df["price"].quantile([1 / 3, 2 / 3])  #I'm doing this because tertile thresholds must match the processed sample for tier metadata.
category_levels = {c: sorted(df[c].dropna().unique().tolist()) for c in ["manufacturer", "fuel", "transmission", "drive", "type"]}  #I'm doing this because sorted unique levels document valid dropdown values for an app.

bundle = {  #I'm doing this because a dict bundles the pipeline with metadata needed at inference time.
    "pipeline": models["rf"],  #I'm doing this because the trained random forest pipeline is what we deploy for price prediction.
    "feature_cols": FEATURES,  #I'm doing this because the app must feed columns in the same order as training.
    "target": TARGET,  #I'm doing this because downstream code can assert it is predicting price.
    "tier_cutoffs": (float(q1), float(q2)),  #I'm doing this because classification uses the same cutoffs for consistent tier labels.
    "tier_labels": ("budget", "mid", "premium"),  #I'm doing this because explicit label order matches confusion matrix ordering.
    "age_base": int(df["year"].max() + 1),  #I'm doing this because inference recomputes vehicle_age with the same anchor year.
    "category_levels": category_levels,  #I'm doing this because UI validation can restrict inputs to seen categories.
}  #I'm doing this because closing the dict completes the artifact payload.

joblib.dump(bundle, out, compress=3)  #I'm doing this because compressed joblib keeps models under GitHub's 100MB limit and still loads in the Streamlit app.
print("saved", out.resolve())  #I'm doing this because printing the absolute path confirms the bundle location.


saved /Users/jahbrae/Downloads/NEW/models/regression_model.pkl


### Important note

This notebook saves the regression bundle under `NEW/models` so it matches the coursework flow in your NEW folder. Notebook 01 is the source of truth for `cleaned_data.csv`; rerun it if upstream cleaning changes.

### Best model choice

I’m picking **RandomForest** here because it had the best RMSE on my holdout among the things I tried, without me hand-tuning for hours. GBR was close; linear/ridge were clearly capped.
